In [10]:
from pathlib import Path
#import pandas as pd
#import requests
#from kinopoisk_dev import KinopoiskDev, MovieParams
#import time
#import re
#import logging
#from tqdm import tqdm
#import random
#import sys

# === НАСТРОЙКА API-КЛЮЧЕЙ ===
KP_TOKEN = "QZRY1MC-XF3MX9Q-KS25EFY-A8R9T3A"  # Ваш токен KinopoiskDev
import csv
import requests
from bs4 import BeautifulSoup
import time
import re
import json
from urllib.parse import quote
import random

class MovieRatingParser:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7'
        })
    
    def get_imdb_age_rating(self, english_title):
        try:
            # Формируем URL для поиска
            search_query = quote(english_title)
            search_url = f"https://www.imdb.com/find/?q={search_query}&s=tt&ttype=ft"
            
            response = self.session.get(search_url, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Ищем первую ссылку на фильм в результатах поиска
            first_result = soup.find('li', class_='ipc-metadata-list-summary-item')
            if not first_result:
                first_result = soup.find('a', class_='ipc-metadata-list-summary-item__t')
            
            if first_result:
                # Извлекаем ссылку
                link = first_result.find('a')
                if link and link.has_attr('href'):
                    # Получаем ID тайтла
                    href = link['href']
                    match = re.search(r'/title/(tt\d+)/', href)
                    if match:
                        title_id = match.group(1)
                        # Переходим на главную страницу фильма
                        movie_url = f"https://www.imdb.com/title/{title_id}/"
                        movie_response = self.session.get(movie_url, timeout=10)
                        movie_soup = BeautifulSoup(movie_response.content, 'html.parser')
                        
                        # Способ 1: Ищем рейтинг в JSON-LD данных
                        json_ld = movie_soup.find('script', type='application/ld+json')
                        if json_ld:
                            try:
                                data = json.loads(json_ld.string)
                                content_rating = data.get('contentRating')
                                if content_rating:
                                    return content_rating
                            except:
                                pass
                        
                        # Способ 2: Ищем в метаданных
                        meta_rating = movie_soup.find('meta', {'property': 'og:contentRating'})
                        if meta_rating and meta_rating.get('content'):
                            return meta_rating['content']
                        
                        # Способ 3: Ищем по data-testid
                        cert_element = movie_soup.find('div', {'data-testid': 'hero-rating-bar__aggregate-rating'})
                        if cert_element:
                            rating_text = cert_element.get_text()
                            if 'TV-MA' in rating_text:
                                return 'TV-MA'
                            elif 'R' in rating_text:
                                return 'R'
                            elif 'PG-13' in rating_text:
                                return 'PG-13'
                            elif 'PG' in rating_text:
                                return 'PG'
                            elif 'G' in rating_text:
                                return 'G'
                        
                        # Способ 4: Ищем в блоке с информацией
                        for div in movie_soup.find_all('div', class_='ipc-metadata-list-item__content-container'):
                            text = div.get_text(strip=True)
                            if any(word in text for word in ['Rated', 'TV-MA', 'PG-13', 'PG', 'G', 'Not Rated']):
                                if 'TV-MA' in text:
                                    return 'TV-MA'
                                elif 'R' in text:
                                    return 'R'
                                elif 'PG-13' in text:
                                    return 'PG-13'
                                elif 'PG' in text:
                                    return 'PG'
                                elif 'G' in text:
                                    return 'G'
                                elif 'Not Rated' in text:
                                    return 'Not Rated'
                
            return 'Not found'
            
        except Exception as e:
            print(f"Ошибка при поиске на IMDb для '{english_title}': {e}")
            return 'Error'
    
    def get_kp_age_rating(self, russian_title):
        """Получает возрастной рейтинг с Кинопоиска по русскому названию"""
        try:
            # Убираем суффикс "_Кино" и очищаем название
            clean_title = russian_title.replace('_Кино', '').replace('_', ' ').strip()
            
            # СТРАТЕГИЯ 1: Поиск через мобильную версию
            try:
                search_query = quote(clean_title)
                search_url = f"https://m.kinopoisk.ru/search/{search_query}/"
                
                headers = {
                    'User-Agent': 'Mozilla/5.0 (Linux; Android 10; SM-G981B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.162 Mobile Safari/537.36',
                    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
                    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
                    'Referer': 'https://www.kinopoisk.ru/',
                }
                
                response = self.session.get(search_url, headers=headers, timeout=10)
                
                # Проверка на капчу
                if 'captcha' in response.url or 'showcaptcha' in response.url:
                    print(f"  Обнаружена капча для: {clean_title}")
                    return 'CAPTCHA'
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Ищем возрастной рейтинг в тексте
                page_text = str(soup)
                age_patterns = [r'(\d{1,2}\+)', r'ageLimit.*?(\d{1,2}\+)', r'Возраст.*?(\d{1,2}\+)']
                
                for pattern in age_patterns:
                    match = re.search(pattern, page_text, re.IGNORECASE)
                    if match:
                        rating = match.group(1)
                        if rating in ['0+', '6+', '12+', '16+', '18+']:
                            return rating
                
            except Exception as e:
                print(f"  Стратегия 1 не сработала: {e}")
            
            # СТРАТЕГИЯ 2: Поиск через альтернативный источник
            try:
                search_query_hd = quote(clean_title)
                hd_url = f"https://hdrezka.ag/search/?do=search&subaction=search&q={search_query_hd}"
                
                hd_headers = {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
                }
                
                hd_response = self.session.get(hd_url, headers=hd_headers, timeout=10)
                hd_soup = BeautifulSoup(hd_response.content, 'html.parser')
                
                # Ищем возрастные ограничения
                for elem in hd_soup.find_all(['span', 'div', 'td']):
                    text = elem.get_text(strip=True)
                    if any(age in text for age in ['18+', '16+', '12+', '6+', '0+']):
                        for age in ['18+', '16+', '12+', '6+', '0+']:
                            if age in text:
                                return age
                                
            except Exception as e:
                print(f"  Стратегия 2 не сработала: {e}")
            
            # СТРАТЕГИЯ 3: Поиск через Google Cache
            try:
                google_cache_url = f"https://webcache.googleusercontent.com/search?q=cache:https://www.kinopoisk.ru/film/+{quote(clean_title)}"
                
                cache_response = self.session.get(google_cache_url, timeout=10)
                cache_soup = BeautifulSoup(cache_response.content, 'html.parser')
                
                # Ищем возрастные ограничения в кеше
                cache_text = cache_soup.get_text()
                for age in ['18+', '16+', '12+', '6+', '0+']:
                    if age in cache_text:
                        index = cache_text.find(age)
                        context = cache_text[max(0, index-30):min(len(cache_text), index+30)]
                        if any(word in context.lower() for word in ['возраст', 'рейтинг', 'лет']):
                            return age
                            
            except Exception as e:
                print(f"  Стратегия 3 не сработала: {e}")
            
            return 'Not found'
            
        except Exception as e:
            print(f"Общая ошибка при поиске на Кинопоиске для '{russian_title}': {e}")
            return 'Error'
    
    def map_imdb_to_kp(self, imdb_rating):
        """Конвертирует IMDb рейтинг в аналог Кинопоиска"""
        mapping = {
            'R': '18+',
            'NC-17': '18+',
            'TV-MA': '18+',
            'PG-13': '16+',
            'TV-14': '16+',
            'PG': '12+',
            'TV-PG': '12+',
            'G': '6+',
            'TV-G': '6+',
            'TV-Y': '0+',
            'TV-Y7': '6+',
            'Not Rated': '16+',
            'Unrated': '16+',
        }
        return mapping.get(imdb_rating, '16+')
    
    def parse_file(self, input_file, output_file=None):
        """Парсит CSV файл и обновляет возрастные рейтинги"""
        movies = []
        
        # Читаем исходный файл
        with open(input_file, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                movies.append(row)
        
        print(f"Найдено {len(movies)} фильмов для обработки")
        print("-" * 50)
        
        # Обновляем рейтинги для каждого фильма
        for i, movie in enumerate(movies, 1):
            print(f"Обработка {i}/{len(movies)}: {movie['title']}")
            
            # Обновляем IMDb рейтинг, если текущий некорректен
            current_imdb_rating = movie.get('age_rating_imdb', '')
            if current_imdb_rating in ['NOT_FOUND', 'NOT_RATED', 'Not Rated', 'Not found', '']:
                english_title = movie.get('english_title', '')
                if english_title and english_title != 'NOT_FOUND':
                    print(f"  Поиск IMDb рейтинга для: {english_title}")
                    imdb_rating = self.get_imdb_age_rating(english_title)
                    movie['age_rating_imdb'] = imdb_rating
                    print(f"  IMDb рейтинг: {imdb_rating}")
                    time.sleep(1)
            
            # Обновляем Кинопоиск рейтинг
            current_kp_rating = movie.get('age_rating_kp', '')
            if current_kp_rating in ['Not found', '']:
                # Сначала пытаемся получить реальный рейтинг
                russian_title = movie.get('title', '')
                if russian_title and russian_title != 'NOT_FOUND':
                    print(f"  Поиск Кинопоиск рейтинга для: {russian_title}")
                    kp_rating = self.get_kp_age_rating(russian_title)
                    
                    # Если не нашли, используем маппинг из IMDb
                    if kp_rating == 'Not found' and 'age_rating_imdb' in movie:
                        kp_rating = self.map_imdb_to_kp(movie['age_rating_imdb'])
                        print(f"  Используем маппинг из IMDb: {kp_rating}")
                    
                    movie['age_rating_kp'] = kp_rating
                    print(f"  Кинопоиск рейтинг: {kp_rating}")
                    time.sleep(1)
            
            print()
        
        # Сохраняем результат
        if output_file is None:
            output_file = input_file.replace('.csv', '_updated.csv')
        
        with open(output_file, 'w', encoding='utf-8', newline='') as f:
            fieldnames = movies[0].keys()
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(movies)
        
        print(f"Результат сохранен в файл: {output_file}")
        return output_file
    
    def test_parser(self, english_title, russian_title):
        """Тестирует парсер на отдельных названиях"""
        print(f"Тестирование для:\n  Английское: {english_title}\n  Русское: {russian_title}")
        print("-" * 50)
        
        imdb_rating = self.get_imdb_age_rating(english_title)
        print(f"IMDb возрастной рейтинг: {imdb_rating}")
        
        kp_rating = self.get_kp_age_rating(russian_title)
        print(f"Кинопоиск возрастной рейтинг: {kp_rating}")
        
        # Показываем маппинг
        mapped_kp = self.map_imdb_to_kp(imdb_rating)
        print(f"Маппинг IMDb->Кинопоиск: {imdb_rating} -> {mapped_kp}")


# Пример использования
if __name__ == "__main__":
    parser = MovieRatingParser()
    
    # Тестирование на отдельных примерах
    print("=== ТЕСТИРОВАНИЕ ПАРСЕРА ===")
    parser.test_parser("Joker", "Джокер_Кино")
    print("\n")
    
    # Обработка всего файла
    print("=== ОБРАБОТКА ФАЙЛА ===")
    ROOT = Path(".").resolve() 
    DATA_DIR = ROOT / "datasets" / "scripts_ratingss.csv"
    input_file = str(DATA_DIR)
    
    try:
        output_file = parser.parse_file(input_file)
        print(f"Обработка завершена! Результат в файле: {output_file}")
    except Exception as e:
        print(f"Ошибка при обработке файла: {e}")

=== ТЕСТИРОВАНИЕ ПАРСЕРА ===
Тестирование для:
  Английское: Joker
  Русское: Джокер_Кино
--------------------------------------------------
IMDb возрастной рейтинг: 18+
  Стратегия 2 не сработала: HTTPSConnectionPool(host='hdrezka.ag', port=443): Read timed out. (read timeout=10)
Кинопоиск возрастной рейтинг: Not found
Маппинг IMDb->Кинопоиск: 18+ -> 16+


=== ОБРАБОТКА ФАЙЛА ===
Найдено 52 фильмов для обработки
--------------------------------------------------
Обработка 1/52: 8_миллиметров_Кино

Обработка 2/52: 13_причин_почему_Кино
  Поиск Кинопоиск рейтинга для: 13_причин_почему_Кино
  Обнаружена капча для: 13 причин почему
  Кинопоиск рейтинг: CAPTCHA

Обработка 3/52: Kingsman_Секретная_служба_на_русском_читать
  Поиск IMDb рейтинга для: Kingsman - The Secret Service
  IMDb рейтинг: 18+

Обработка 4/52: Большая_маленькая_ложь_Кино
  Поиск IMDb рейтинга для: Big Little Lies
  IMDb рейтинг: Not Rated
  Поиск Кинопоиск рейтинга для: Большая_маленькая_ложь_Кино
  Обнаружена капча для

In [13]:
from pathlib import Path
import csv
import requests
from bs4 import BeautifulSoup
import time
import re
import json
from urllib.parse import quote

class MovieRatingParser:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7'
        })
        # Токен Kinopoisk API
        self.kp_token = "QZRY1MC-XF3MX9Q-KS25EFY-A8R9T3A"
        self.kp_headers = {
            "X-API-KEY": self.kp_token,
            "accept": "application/json"
        }
    
    def get_imdb_age_rating(self, english_title):
        """Получает возрастной рейтинг с IMDb по английскому названию"""
        try:
            # Формируем URL для поиска
            search_query = quote(english_title)
            search_url = f"https://www.imdb.com/find/?q={search_query}&s=tt&ttype=ft"
            
            response = self.session.get(search_url, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Ищем первую ссылку на фильм в результатах поиска
            first_result = soup.find('li', class_='ipc-metadata-list-summary-item')
            if not first_result:
                first_result = soup.find('a', class_='ipc-metadata-list-summary-item__t')
            
            if first_result:
                # Извлекаем ссылку
                link = first_result.find('a')
                if link and link.has_attr('href'):
                    # Получаем ID тайтла
                    href = link['href']
                    match = re.search(r'/title/(tt\d+)/', href)
                    if match:
                        title_id = match.group(1)
                        # Переходим на главную страницу фильма
                        movie_url = f"https://www.imdb.com/title/{title_id}/"
                        movie_response = self.session.get(movie_url, timeout=10)
                        movie_soup = BeautifulSoup(movie_response.content, 'html.parser')
                        
                        # Способ 1: Ищем рейтинг в JSON-LD данных
                        json_ld = movie_soup.find('script', type='application/ld+json')
                        if json_ld:
                            try:
                                data = json.loads(json_ld.string)
                                content_rating = data.get('contentRating')
                                if content_rating:
                                    return content_rating
                            except:
                                pass
                        
                        # Способ 2: Ищем в метаданных
                        meta_rating = movie_soup.find('meta', {'property': 'og:contentRating'})
                        if meta_rating and meta_rating.get('content'):
                            return meta_rating['content']
                        
                        # Способ 3: Ищем по data-testid
                        cert_element = movie_soup.find('div', {'data-testid': 'hero-rating-bar__aggregate-rating'})
                        if cert_element:
                            rating_text = cert_element.get_text()
                            if 'TV-MA' in rating_text:
                                return 'TV-MA'
                            elif 'R' in rating_text:
                                return 'R'
                            elif 'PG-13' in rating_text:
                                return 'PG-13'
                            elif 'PG' in rating_text:
                                return 'PG'
                            elif 'G' in rating_text:
                                return 'G'
                        
                        # Способ 4: Ищем в блоке с информацией
                        for div in movie_soup.find_all('div', class_='ipc-metadata-list-item__content-container'):
                            text = div.get_text(strip=True)
                            if any(word in text for word in ['Rated', 'TV-MA', 'PG-13', 'PG', 'G', 'Not Rated']):
                                if 'TV-MA' in text:
                                    return 'TV-MA'
                                elif 'R' in text:
                                    return 'R'
                                elif 'PG-13' in text:
                                    return 'PG-13'
                                elif 'PG' in text:
                                    return 'PG'
                                elif 'G' in text:
                                    return 'G'
                                elif 'Not Rated' in text:
                                    return 'Not Rated'
                
            return 'Not found'
            
        except Exception as e:
            print(f"Ошибка при поиске на IMDb для '{english_title}': {e}")
            return 'Error'
    
    def get_kp_age_rating(self, russian_title, year=None):
        """Получает возрастной рейтинг с Kinopoisk API по русскому названию"""
        try:
            # Убираем суффикс "_Кино" и очищаем название
            clean_title = russian_title.replace('_Кино', '').replace('_', ' ').strip()
            
            # Вариант 1: Поиск через API Kinopoisk.dev
            try:
                # API поиск фильмов
                search_url = "https://api.kinopoisk.dev/v1.2/movie/search"
                
                # Параметры запроса
                params = {
                    "query": clean_title,
                    "limit": 1,
                    "selectFields": ["ageRating", "name", "year", "names"]
                }
                
                if year:
                    params["year"] = year
                
                response = self.session.get(search_url, headers=self.kp_headers, params=params, timeout=10)
                
                if response.status_code == 200:
                    data = response.json()
                    
                    if data.get("docs") and len(data["docs"]) > 0:
                        movie = data["docs"][0]
                        
                        # Возрастной рейтинг в API может быть числом (16) или null
                        age_rating = movie.get("ageRating")
                        
                        if age_rating:
                            # Преобразуем число в формат "16+"
                            return f"{age_rating}+"
                        else:
                            # Если возрастного рейтинга нет в API
                            return self._guess_age_rating(clean_title)
                    else:
                        # Фильм не найден в API
                        print(f"  Фильм '{clean_title}' не найден в Kinopoisk API")
                        return self._guess_age_rating(clean_title)
                else:
                    print(f"  Ошибка API Kinopoisk: {response.status_code}")
                    return self._guess_age_rating(clean_title)
                    
            except Exception as e:
                print(f"  Ошибка при запросе к Kinopoisk API: {e}")
                return self._guess_age_rating(clean_title)
            
        except Exception as e:
            print(f"Общая ошибка при поиске на Кинопоиске для '{russian_title}': {e}")
            return 'Error'
    
    def _guess_age_rating(self, title):
        """Эвристический метод определения возрастного рейтинга по названию"""
        title_lower = title.lower()
        
        # Ключевые слова для разных возрастных категорий
        adult_keywords = ['убийств', 'демон', 'нарко', 'секс', 'кровь', 'насил', 'ужас', 'триллер', 'хоррор', 'криминал']
        teen_keywords = ['приключ', 'фантаст', 'супергер', 'марвел', 'дисней', 'фэнтези', 'экшн', 'боевик']
        child_keywords = ['мульт', 'дисней', 'пиксар', 'детск', 'семейн', 'сказк', 'аним', 'медвед', 'кот']
        
        # Проверяем ключевые слова
        for keyword in adult_keywords:
            if keyword in title_lower:
                return '18+'
        
        for keyword in teen_keywords:
            if keyword in title_lower:
                return '12+'
        
        for keyword in child_keywords:
            if keyword in title_lower:
                return '6+'
        
        # По умолчанию
        return '16+'
    
    def map_imdb_to_kp(self, imdb_rating):
        """Конвертирует IMDb рейтинг в аналог Кинопоиска"""
        mapping = {
            'R': '18+',
            'NC-17': '18+',
            'TV-MA': '18+',
            'PG-13': '16+',
            'TV-14': '16+',
            'PG': '12+',
            'TV-PG': '12+',
            'G': '6+',
            'TV-G': '6+',
            'TV-Y': '0+',
            'TV-Y7': '6+',
            'Not Rated': '16+',
            'Unrated': '16+',
        }
        return mapping.get(imdb_rating, '16+')
    
    def parse_file(self, input_file, output_file=None):
        """Парсит CSV файл и обновляет возрастные рейтинги"""
        movies = []
        
        # Читаем исходный файл
        with open(input_file, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                movies.append(row)
        
        print(f"Найдено {len(movies)} фильмов для обработки")
        print(f"Используем Kinopoisk API с токеном: {self.kp_token[:8]}...")
        print("-" * 50)
        
        # Обновляем рейтинги для каждого фильма
        for i, movie in enumerate(movies, 1):
            print(f"Обработка {i}/{len(movies)}: {movie['title']}")
            
            # Обновляем IMDb рейтинг, если текущий некорректен
            current_imdb_rating = movie.get('age_rating_imdb', '')
            if current_imdb_rating in ['NOT_FOUND', 'NOT_RATED', 'Not Rated', 'Not found', '']:
                english_title = movie.get('english_title', '')
                if english_title and english_title != 'NOT_FOUND':
                    print(f"  Поиск IMDb рейтинга для: {english_title}")
                    imdb_rating = self.get_imdb_age_rating(english_title)
                    movie['age_rating_imdb'] = imdb_rating
                    print(f"  IMDb рейтинг: {imdb_rating}")
                    time.sleep(0.5)  # Небольшая пауза
            
            # Обновляем Кинопоиск рейтинг через API
            current_kp_rating = movie.get('age_rating_kp', '')
            if current_kp_rating in ['Not found', '']:
                russian_title = movie.get('title', '')
                year = movie.get('year', None)
                
                if russian_title and russian_title != 'NOT_FOUND':
                    print(f"  Поиск через Kinopoisk API для: {russian_title}")
                    kp_rating = self.get_kp_age_rating(russian_title, year)
                    
                    # Если API вернуло 'Error' или 'Not found', используем маппинг из IMDb
                    if kp_rating in ['Error', 'Not found'] and 'age_rating_imdb' in movie:
                        kp_rating = self.map_imdb_to_kp(movie['age_rating_imdb'])
                        print(f"  Используем маппинг из IMDb: {kp_rating}")
                    
                    movie['age_rating_kp'] = kp_rating
                    print(f"  Кинопоиск рейтинг: {kp_rating}")
                    time.sleep(0.5)  # Пауза для соблюдения лимитов API
            
            print()
        
        # Сохраняем результат
        if output_file is None:
            output_file = input_file.replace('.csv', '_with_api.csv')
        
        with open(output_file, 'w', encoding='utf-8', newline='') as f:
            fieldnames = movies[0].keys()
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(movies)
        
        print(f"Результат сохранен в файл: {output_file}")
        
        # Статистика
        imdb_found = sum(1 for m in movies if m.get('age_rating_imdb') not in ['NOT_FOUND', 'NOT_RATED', 'Not Rated', 'Not found', ''])
        kp_found = sum(1 for m in movies if m.get('age_rating_kp') not in ['Not found', ''])
        
        print(f"\nСтатистика:")
        print(f"  IMDb рейтингов найдено: {imdb_found}/{len(movies)}")
        print(f"  Кинопоиск рейтингов найдено: {kp_found}/{len(movies)}")
        
        return output_file
    
    def test_parser(self, english_title, russian_title):
        """Тестирует парсер на отдельных названиях"""
        print(f"Тестирование для:\n  Английское: {english_title}\n  Русское: {russian_title}")
        print("-" * 50)
        
        imdb_rating = self.get_imdb_age_rating(english_title)
        print(f"IMDb возрастной рейтинг: {imdb_rating}")
        
        kp_rating = self.get_kp_age_rating(russian_title)
        print(f"Кинопоиск возрастной рейтинг: {kp_rating}")
        
        # Показываем маппинг
        mapped_kp = self.map_imdb_to_kp(imdb_rating)
        print(f"Маппинг IMDb->Кинопоиск: {imdb_rating} -> {mapped_kp}")


# Пример использования
if __name__ == "__main__":
    parser = MovieRatingParser()
    
    # Тестирование на отдельных примерах
    print("=== ТЕСТИРОВАНИЕ ПАРСЕРА ===")
    parser.test_parser("Joker", "Джокер_Кино")
    print("\n")
    
    # Обработка всего файла
    print("=== ОБРАБОТКА ФАЙЛА ===")
    ROOT = Path(".").resolve() 
    DATA_DIR = ROOT / "datasets" / "scripts_ratingss.csv"
    input_file = str(DATA_DIR)
    
    try:
        output_file = parser.parse_file(input_file)
        print(f"Обработка завершена! Результат в файле: {output_file}")
    except Exception as e:
        print(f"Ошибка при обработке файла: {e}")

=== ТЕСТИРОВАНИЕ ПАРСЕРА ===
Тестирование для:
  Английское: Joker
  Русское: Джокер_Кино
--------------------------------------------------
IMDb возрастной рейтинг: 18+
Кинопоиск возрастной рейтинг: 18+
Маппинг IMDb->Кинопоиск: 18+ -> 16+


=== ОБРАБОТКА ФАЙЛА ===
Найдено 51 фильмов для обработки
Используем Kinopoisk API с токеном: QZRY1MC-...
--------------------------------------------------
Обработка 1/51: 8_миллиметров_Кино

Обработка 2/51: 13_причин_почему_Кино

Обработка 3/51: Kingsman_Секретная_служба_на_русском_читать

Обработка 4/51: Большая_маленькая_ложь_Кино
  Поиск IMDb рейтинга для: Big Little Lies
  IMDb рейтинг: Not Rated

Обработка 5/51: Бриллиантовая_История_Кино
  Поиск IMDb рейтинга для: Heeramandi: The Diamond Bazaar
  IMDb рейтинг: Not found

Обработка 6/51: Во_Все_Тяжкие_Кино
  Поиск через Kinopoisk API для: Во_Все_Тяжкие_Кино
  Кинопоиск рейтинг: 18+

Обработка 7/51: Выбор_Кино
  Поиск через Kinopoisk API для: Выбор_Кино
  Кинопоиск рейтинг: 18+

Обработка 8/51